# Puffin Transcription Initiation

Puffin predicts per-base transcription initiation signals from DNA sequence and decomposes them into learned motif contributions ([Dudnyk et al., *Science* 2024](https://doi.org/10.1126/science.adj0116)). Both tools share the same model: `puffin-prediction` is a fast forward pass returning all 10 channels, and `puffin-interpretation` runs gradient-based motif decomposition for one chosen target signal and strand.

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools, display_doc_link, display_docs_section, display_overview

display_doc_link("puffin")
display_overview("puffin")
display_docs_section("puffin", "Background")

## Available tools

In [ ]:
display_available_tools("puffin")

### `run_puffin_prediction`

Fast forward pass through Puffin. For each input DNA sequence (>= 651 bp) returns per-base predictions across 10 channels (five transcription assays on both strands). Per-base output length equals `len(sequence) - 650`.

In [ ]:
from proto_tools import (
    PuffinPredictionConfig,
    PuffinPredictionInput,
    TRACK_NAMES,
    run_puffin_prediction,
)

In [ ]:
# Display input docs
display_api_reference("puffin-prediction", "input", "run_puffin_prediction")

In [ ]:
# Display config docs
display_api_reference("puffin-prediction", "config", "run_puffin_prediction")

In [ ]:
# 1650 bp input -> 1000 bp per-base output
sequence = "ATCG" * 412 + "AT"
prediction_result = run_puffin_prediction(
    PuffinPredictionInput(sequences=[sequence]),
    PuffinPredictionConfig(),
)

entry = prediction_result.results[0]
fantom_idx = TRACK_NAMES.index("FANTOM_CAGE+")
peak = max(range(entry.output_length), key=lambda i: entry.predictions[i][fantom_idx])
print(f"shape: {entry.output_length} bp x {len(entry.predictions[0])} channels")
print(f"FANTOM_CAGE+ peak at input position {entry.output_start + peak}")

In [ ]:
display_api_reference("puffin-prediction", "output", "run_puffin_prediction")

### `run_puffin_interpretation`

Gradient-based motif decomposition for one chosen target signal and strand. Returns the per-base prediction plus 18 motif activation tracks (9 motifs x 2 strands) and basepair-level contribution scores.

In [ ]:
from proto_tools import (
    MOTIF_NAMES,
    PuffinInterpretationConfig,
    PuffinInterpretationInput,
    run_puffin_interpretation,
)

In [ ]:
# Display input docs
display_api_reference("puffin-interpretation", "input", "run_puffin_interpretation")

In [ ]:
# Display config docs
display_api_reference("puffin-interpretation", "config", "run_puffin_interpretation")

In [ ]:
# Decompose the PRO_CAP forward-strand prediction
sequence = "ATCG" * 412 + "AT"
interpretation_result = run_puffin_interpretation(
    PuffinInterpretationInput(sequences=[sequence]),
    PuffinInterpretationConfig(target_signal="PRO_CAP"),
)

r = interpretation_result.results[0]
print(f"target_signal: {interpretation_result.target_signal}, reverse_strand: {interpretation_result.reverse_strand}")
print(f"motif tracks: {len(r.motif_activations)} (9 motifs x 2 strands)")
print(f"motifs: {MOTIF_NAMES}")

In [ ]:
display_api_reference("puffin-interpretation", "output", "run_puffin_interpretation")

## Export Results

In [ ]:
from pathlib import Path

out_dir = Path("puffin_output")
out_dir.mkdir(exist_ok=True)
prediction_result.export(name="prediction", export_path=out_dir, file_format="json")
interpretation_result.export(name="interpretation", export_path=out_dir, file_format="json")